In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import zCurve as z
import hilbert as h
from dtaidistance import dtw
import seaborn as sns

sns.set_context("paper", font_scale=1.3)
sns.set_context('notebook')
sns.set_palette("deep")
sns.set_theme(font='sans-serif', font_scale=1.3)
sns.set_style('whitegrid')
deepBlue = sns.color_palette('deep')[0]
deepOrange = sns.color_palette('deep')[1]
deepGreen = sns.color_palette('deep')[2]
deepRed = sns.color_palette('deep')[3]

## Utility functions

In [ ]:
def get_files_in_folder(folder_path):
    try:
        data_files = [folder_path + "/" + f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
        data_files_copy = data_files.copy()
        for file_path in data_files_copy:
            if ".csv" not in file_path:
                data_files.remove(file_path)
        return data_files
    except Exception as e:
        print(f"An error occurred while getting files in the folder: {str(e)}")
        return []

def plot_heatmap(value_matrix, column_names, title_text, column_classes=None, display_file_names=True):

    """
    Plot heatmap of distances/measures for easy visualization.
    Parameters
    ----------
    value_matrix: matrix with pairwise distances
    column_names: list of column (file) names
    title_text: title of the figure

    """

    fig, ax = plt.subplots(figsize=(10,10))
    im = ax.imshow(value_matrix, origin='lower', cmap='seismic')
    
    if display_file_names:
        # Show all ticks and label them with the respective list entries
        ax.set_xticks(range(len(column_names)), labels=column_names,
                        rotation=45, ha="right", rotation_mode="anchor")
        ax.set_yticks(range(len(column_names)), labels=column_names)

        # Color labels based on class
        if column_classes:
            ylabels = ax.get_yticklabels()
            xlabels = ax.get_xticklabels()
            for i, (xlabel, ylabel) in enumerate(zip(xlabels, ylabels)):
                class_label = column_classes[i]
                xlabel.set_color(sns.color_palette('tab20')[class_label])
                ylabel.set_color(sns.color_palette('tab20')[class_label])
    else: 
        # Show groups numbers instead of all individual file names
        
        n_groups = len(np.unique(column_classes))
        cummulative_sum = 0
        group_label_positions = np.zeros(n_groups)

        
        for i in range(n_groups):
            idx = (np.array(column_classes) == i)
   
            n_cases = np.sum(idx.astype(int))
            group_label_positions[i] = cummulative_sum + n_cases / 2
            cummulative_sum = cummulative_sum + n_cases

            if i < (n_groups - 1):
                ax.axhline(y=cummulative_sum - 0.5, color='white')
                ax.axvline(x=cummulative_sum - 0.5, color='white')

        ax.set_xticks(group_label_positions, labels=[f'{i+1}' for i in range(n_groups)], fontweight='bold')
        ax.set_yticks(group_label_positions, labels=[f'{i+1}' for i in range(n_groups)], fontweight='bold')
        ax.grid(visible=False)

    #ax.invert_yaxis()
    #ax.invert_xaxis()

    fig.colorbar(im, shrink=0.6)
    ax.set_title(title_text)

    fig.tight_layout()
    plt.show()
    return fig

## Pre-processing functions

In [ ]:
def zNormalize_csp(csp):
    """ Standarize csp to have mean 0 and standard deviation of 1 """
    return (csp - np.mean(csp)) / np.std(csp)

def modified_zscore(csp):
    median = np.median(csp)
    mad = np.median(np.abs(csp - median))
    modified_z_scores = 0.6745 * (csp - median) / mad
    return modified_z_scores 

def shift_csp(csp, k):    
    csp_shifted = np.copy(csp)
    ds = csp[1:] - csp[:-1]

    ds_std = k * np.std(ds)
    threshold = ds_std
    
    for i, ds_i in enumerate(ds):
        if np.abs(ds_i) > threshold:
            csp_shifted[i+1:] = csp_shifted[i+1:] - ds_i * (1 - threshold / np.abs(ds_i))
    return csp_shifted


## Metric Functions

In [ ]:
def euclidean_distance(csp1, csp2, normalization=False):
    if normalization == 'zscore':
        csp1 = zNormalize_csp(csp1)
        csp2 = zNormalize_csp(csp2)
    elif normalization == 'modified_zscore':
        csp1 = modified_zscore(csp1)
        csp2 = modified_zscore(csp2)
    difference = np.sqrt(np.dot(csp1 - csp2, csp1 - csp2))
    return difference

def dynamictimewarping_distance(csp1, csp2, normalization=False):
    if normalization == 'zscore':
        csp1 = zNormalize_csp(csp1)
        csp2 = zNormalize_csp(csp2)
    elif normalization == 'modified_zscore':
        csp1 = modified_zscore(csp1)
        csp2 = modified_zscore(csp2)

    difference = dtw.distance(csp1, csp2) / (len(csp1) + len(csp2))
    return difference

def pearson_correlation(csp1, csp2):
    csp1_mod = csp1 - np.mean(csp1)
    csp2_mod = csp2 - np.mean(csp2)
    corr = np.dot(csp1_mod,csp2_mod) / np.sqrt(np.dot(csp1_mod,csp1_mod) * np.dot(csp2_mod,csp2_mod))
    return corr

## Load files with synthetic data

In [ ]:
# Get files
parentFolder = 'results/synthetic_lanechangecitos_v7'
data_files = get_files_in_folder(folder_path=parentFolder)
data_files.sort()
data_files.reverse()

# Remove csv files from previous runs
data_files = [file_path for file_path in data_files.copy() if 'similarity' not in file_path]
truncate_end = None 

## Automatically assign offset and scale for the speed and steering angle

In [ ]:
# Decide offset and scale factor for the steering angle such that 
# the mean and standard deviation is rougly the same for both the 
# speed and steering angle. 
#
# Note that the speed is "dominating"

speed_means, speed_stds = [], []
steering_angle_means, steering_angle_stds = [], []

for i, file_path in enumerate(data_files):
    data = pd.read_csv(file_path, sep=";")
    speeds = np.array(data['Speed (m/s)'])[:truncate_end]
    steering_angles = np.array(data['Steering angle (deg)'])[:truncate_end] 
    speed_means.append(np.mean(speeds))
    steering_angle_means.append(np.mean(steering_angles))
    speed_stds.append(np.std(speeds))
    steering_angle_stds.append(np.std(steering_angles))

print('BEFORE OFFSET AND SCALING\n-------------------------')
print('The average mean and standard deviation for... ')
print(f'Speed : {np.mean(speed_means):.4f} {np.mean(speed_stds):.4f}')
print(f'Steering angle : {np.mean(steering_angle_means):.4f} {np.mean(steering_angle_stds):.4f}\n')


# Set offset and scale to 100 and 1, respectively, for the speed
offsets_dict, scales_dict = {"Speed (m/s)": 100}, {"Speed (m/s)": 1}
speed_means, speed_stds = [], []
for i, file_path in enumerate(data_files):
    data = pd.read_csv(file_path, sep=";")
    speeds = np.array(data['Speed (m/s)'])[:truncate_end] * scales_dict['Speed (m/s)'] + offsets_dict["Speed (m/s)"]
    speed_means.append(np.mean(speeds))
    speed_stds.append(np.std(speeds))

# Set offset and scale for steering angle such that they are approximately the same as for the speeed
scales_dict["Steering angle (deg)"] = round(np.mean(speed_stds) / np.mean(steering_angle_stds))
offsets_dict["Steering angle (deg)"] = round(np.mean(speed_means) - scales_dict["Steering angle (deg)"] * np.mean(steering_angle_means))

# Print out the mean and standard deviation after offset and scaling
speed_means, speed_stds = [], []
steering_angle_means, steering_angle_stds = [], []

for i, file_path in enumerate(data_files):
    data = pd.read_csv(file_path, sep=";")
    speeds = scales_dict["Speed (m/s)"] * np.array(data['Speed (m/s)'])[:truncate_end] + offsets_dict["Speed (m/s)"]
    steering_angles = scales_dict["Steering angle (deg)"] * np.array(data['Steering angle (deg)'])[:truncate_end] + offsets_dict["Steering angle (deg)"]
    speed_means.append(np.mean(speeds))
    steering_angle_means.append(np.mean(steering_angles))
    speed_stds.append(np.std(speeds))
    steering_angle_stds.append(np.std(steering_angles))

print('AFTER OFFSET AND SCALING\n------------------------')
print('The average mean and standard deviation for... ')
print(f'Speed : {np.mean(speed_means):.4f} {np.mean(speed_stds):.4f}')
print(f'Steering angle : {np.mean(steering_angle_means):.4f} {np.mean(steering_angle_stds):.4f}')

## Compute the Morton index

In [ ]:
# Store file basenames and classes for easy plotting
file_basenames = []
file_classes = []

# Compute and add morton index columns to files in data_files
for file_path in data_files:

    print(file_path)

    basename = os.path.splitext(os.path.basename(file_path))[0]
    file_basenames.append(basename)
    category = re.sub("\d","",basename)
    file_classes.append(category)
    data = pd.read_csv(file_path, sep=";")

    n_timestamps = len(data['Speed (m/s)'])
    n_dimensions = len(scales_dict)
    tmp_matrix = np.zeros((n_timestamps, n_dimensions))
    for j, data_key in enumerate(scales_dict.keys()):
        tmp_matrix[:, j] = [int(i) for i in data[data_key] * scales_dict[data_key] + offsets_dict[data_key]]

    # Build Morton (Z-order) Index (if chosen or if building both)
    data['Morton_Index'] = [z.interlace(int(x), int(y), bits_per_dim=10) for x, y in zip(tmp_matrix[:,0], tmp_matrix[:,1])]
    df = pd.DataFrame(data=data)
    df.to_csv(file_path, sep=";", index=False)

    # ===============================
    # Build Hilbert Index
    # ===============================
    data["Hilbert_Index"] = h.encode(np.ascontiguousarray(np.array([tmp_matrix[:,0], tmp_matrix[:,1]]).T), 2, 10)
    df = pd.DataFrame(data=data)
    df.to_csv(file_path, sep=";", index=False)


# Set numeric label for class categories based on their filenames 
file_classes_labels = list(set(file_classes))
file_classes_labels.sort()
file_classes_labels.reverse()

for k, label in enumerate(file_classes_labels):
    for i in range(len(file_classes)):
        if file_classes[i] == label:
            file_classes[i] = k

## Plot the CSPs, downshifted CSPs, speed and steering angle

In [ ]:
downshift_threshold = 1 # In units of the STD of the numerical derivative
visualize, normalize = False, False # If plotting the (un)normalized 
save_figures = False
encoding_type = 'hilbert'

for i, file_path in enumerate(data_files):
    data = pd.read_csv(file_path, sep=";")
    file_basename = os.path.splitext(os.path.basename(file_path))[0]
    file_basename = file_basenames[i]
    print(i, file_basename)

    # Load the orignal CSP and the dowshifted CSP
    if encoding_type == 'morton':
        csp = np.array(data['Morton_Index'])[:truncate_end]
    elif encoding_type == 'hilbert':
        csp = np.array(data['Hilbert_Index'])[:truncate_end]
    
    if downshift_threshold:
        csp_shifted = shift_csp(csp, k=downshift_threshold)
    else:
        csp_shifted = csp

    # Load the transformed speed and steering angles
    steering_angles = np.array(data['Steering angle (deg)']*scales_dict['Steering angle (deg)'] + offsets_dict['Steering angle (deg)'])[:truncate_end]
    speeds = np.array(data['Speed (m/s)']*scales_dict['Speed (m/s)'] + offsets_dict['Speed (m/s)'])[:truncate_end]
    
    # Load the time in seconds passed
    time_array = np.array(data['Time (seconds)'])[:truncate_end] + np.array(data['Time (microseconds)'])[:truncate_end] / 10**6
    
    # Plot the four signals 
    arr_to_plot = [csp, csp_shifted, steering_angles, speeds]
    plot_labels = ['Original CSP', 'Shifted CSP', 'Steering angle', 'Speed']
    if visualize:
        fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6,4))
        for arr_i, label_i in zip(arr_to_plot[2:], plot_labels[2:]):
            if normalize:
                ax.plot(time_array, zNormalize_csp(arr_i), marker='.', ls='-', label=label_i)
            else:
                ax.plot(time_array, arr_i, marker='.', ls='-', label=label_i)
  

        ax.set_title(file_basename + f' : downshift threshold {downshift_threshold}$\\sigma$', fontweight='bold')
        ax.set_xlabel(f'Time (seconds)')
        ax.set_ylabel(f'Standarized Magnitude')
        ax.legend()
        fig.tight_layout()
        if save_figures:
            plt.savefig(os.path.join(parentFolder, file_basename + '_morton.png'))
        plt.show()

## Compute the similarity matrices

In [ ]:
# Compute pairwise distances (euclidean, dtw, ...)
euclidean_distance_matrix = np.zeros((len(data_files), len(data_files)))
pearson_correlation_matrix = np.zeros((len(data_files), len(data_files)))
dtw_distance_matrix = np.zeros((len(data_files), len(data_files)))

# NOTE : Some distance measures are not symmetric
symmetric = True

for i, file_path_i in enumerate(data_files):
    print(f'Row {i+1} out of total {len(data_files)}')
    data = pd.read_csv(file_path_i, sep=";")

    # NOTE : Specific for current file format, only want first lc

    
    if encoding_type == 'morton':
        csp_i = np.array(data['Morton_Index'])[:truncate_end]
    elif encoding_type == 'hilbert':
        csp_i = np.array(data['Hilbert_Index'])[:truncate_end]

    if downshift_threshold:
        csp_i = shift_csp(csp_i, k=downshift_threshold)


    if symmetric:
        j_start = i
    else:
        j_start = 0

    for j in range(j_start, len(data_files)):
        file_path_j = data_files[j]
        data = pd.read_csv(file_path_j, sep=";")
        
        if encoding_type == 'morton':
            csp_j = np.array(data['Morton_Index'])[:truncate_end]
        elif encoding_type == 'hilbert':
            csp_j = np.array(data['Hilbert_Index'])[:truncate_end]
            
        if downshift_threshold:
            csp_j = shift_csp(csp_j, k=downshift_threshold)
        

        # If CSPs are of unequal lengths -> pad the shorter CSP
        padding = len(csp_j) - len(csp_i)
        pad_before = (np.abs(padding) // 2) 
        pad_after = (np.abs(padding) // 2) + (np.abs(padding) % 2)
        if padding > 0:
            csp_i = np.pad(csp_i, pad_width=(pad_before, pad_after), mode='edge')
        elif padding < 0:
            csp_j = np.pad(csp_j, pad_width=(pad_before, pad_after), mode='edge')
        
        # Calculate distances
        ec_dist = euclidean_distance(csp_i, csp_j, normalization='zscore')
        pc_dist = pearson_correlation(csp_i, csp_j) 
        dtw_dist = dynamictimewarping_distance(csp_i, csp_j, normalization='zscore')

        # If the metrics are symmetric -> don't calculate each distance twice !!!
        if symmetric:
            euclidean_distance_matrix[[i,j], [j,i]] = ec_dist
            pearson_correlation_matrix[[i,j], [j,i]] = pc_dist
            dtw_distance_matrix[[i,j], [j,i]] = dtw_dist
        else:
            euclidean_distance_matrix[i, j] = ec_dist
            pearson_correlation_matrix[i, j] = pc_dist
            dtw_distance_matrix[i, j] = dtw_dist

In [ ]:
def normalize_value_matrix(value_matrix, mode='ascending'):
    """
    Perform linear transformation of values in simailrity matrix to range from 0 to 1. 
    Select keyword ascending if high values in value_matrix indicate more similar
    elememts. If the opposite, select descending. 
    """
    if mode == 'ascending':
        a = 1 / (1 - np.max(value_matrix) / np.min(value_matrix))
        b = 1 / (np.max(value_matrix) - np.min(value_matrix))
    elif mode == 'descending':
        a = 1 / (1 - np.min(value_matrix) / np.max(value_matrix))
        b = 1 / (np.min(value_matrix) - np.max(value_matrix))
    else:
        print('Invalid mode')
        return
    
    value_matrix = a + b * value_matrix
    return value_matrix

euclidean_distances_new = normalize_value_matrix(euclidean_distance_matrix, mode='descending')
pearson_correlation_new = normalize_value_matrix(pearson_correlation_matrix, mode='ascending')
dtw_distances_new = normalize_value_matrix(dtw_distance_matrix, mode='descending')

## Plot and save similarity matrices as heatmaps

In [ ]:
common_file_name = f'Heatmap_CSP_scale_{scales_dict['Steering angle (deg)']}_shift{downshift_threshold}_0220.png'
common_title = f' downshift of {downshift_threshold}$\\sigma$'

heatmap_fig = plot_heatmap(euclidean_distances_new, column_names=file_basenames,
                title_text='Euclidean Distance, ' + common_title, column_classes=file_classes, display_file_names=False) 
heatmap_fig.savefig(parentFolder + '/L2_' + common_file_name)

heatmap_fig = plot_heatmap(pearson_correlation_new, column_names=file_basenames,
                title_text='Pearson Correlation, ' + common_title, column_classes=file_classes, display_file_names=False)
heatmap_fig.savefig(parentFolder + '/Pearson_' + common_file_name)

heatmap_fig = plot_heatmap(dtw_distances_new, column_names=file_basenames,
                title_text='DTW, ' + common_title, column_classes=file_classes, display_file_names=False)
heatmap_fig.savefig(parentFolder + '/DTW_' + common_file_name)

## Find the two most best matching classes for each case in the data, for each metric
Matching is based on the average similairty score and results are saved in .csv files

In [ ]:
# Number of distinct classes and measures
n_classes = len(file_classes_labels)
measures = ['L2', 'Pearson', 'DTW']
n_measures = len(measures)

# Dicts to save averages for each case and class
euclidean_averages_dict = {}
pearson_averages_dict = {}
dtw_averages_dict = {}

for i, case_i in enumerate(file_basenames):
    buff = np.zeros((n_measures, n_classes))
    label_i = file_basenames[i]
    

    # Find the average similarity score between case i and the cases within each seperate class
    for j in range(n_classes):
        class_idx = np.where(np.array(file_classes) == j)[0]
        class_idx = [el for el in class_idx if el != i]
        buff[0,j] = np.mean(euclidean_distances_new[i, class_idx])
        buff[1,j] = np.mean(pearson_correlation_new[i, class_idx])
        buff[2,j] = np.mean(dtw_distances_new[i, class_idx])
    
    # Find the 2 most similar classes of maneuvers 
    matching_classes = np.flip(np.argsort(buff, axis=1, )[:,-2:], axis=1).astype(int)

    # For case i, save the 2 best matching classes with their corresponding average similarity scores
    euclidean_averages_dict[label_i] = np.concatenate((matching_classes[0, :], buff[0,:]))
    pearson_averages_dict[label_i] = np.concatenate((matching_classes[1, :], buff[1,:]))
    dtw_averages_dict[label_i] = np.concatenate((matching_classes[2, :], buff[2,:]))


# Create dataframes with similatity scores
df_column_names = ['1st Best', '2nd Best'] + file_classes_labels
euclidean_averages_df = pd.DataFrame.from_dict(euclidean_averages_dict, orient="index", columns=df_column_names)
pearson_averages_df = pd.DataFrame.from_dict(pearson_averages_dict, orient="index", columns=df_column_names)
dtw_averages_df = pd.DataFrame.from_dict(dtw_averages_dict, orient="index", columns=df_column_names)

# Insert column with numeric class label
euclidean_averages_df.insert(0, column='Class Label', value=file_classes)
pearson_averages_df.insert(0, column='Class Label', value=file_classes)
dtw_averages_df.insert(0, column='Class Label', value=file_classes)

# Change dtype for the columns with the best matching classes
euclidean_averages_df = euclidean_averages_df.astype({"1st Best": int, "2nd Best": int})
pearson_averages_df = pearson_averages_df.astype({"1st Best": int, "2nd Best": int})
dtw_averages_df = dtw_averages_df.astype({"1st Best": int, "2nd Best": int})

# Save similarity tables to CSV 
common_file_name = f'best_matches_shift{downshift_threshold}.csv'
euclidean_averages_df.to_csv(parentFolder + '/L2_' + common_file_name, sep=";", index=True, index_label='ID')
pearson_averages_df.to_csv(parentFolder + '/Pearson_' + common_file_name, sep=";", index=True, index_label='ID')
dtw_averages_df.to_csv(parentFolder + '/DTW_' + common_file_name, sep=";", index=True, index_label='ID')

## Evaluate the overall and class-wise performance for each metric

In [ ]:
def calculate_metrics(tp, fp, tn, fn):
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
    accuracy = (tp + tn) / (tp + fp + tn + fn)
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    false_positive_rate = fp / (fp + tn) if (fp + tn) != 0 else 0
    false_negative_rate = fn / (fn + tp) if (fn + tp) != 0 else 0
    mcc_denominator = (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)
    mcc = (tp * tn - fp * fn) / (mcc_denominator ** 0.5) if mcc_denominator != 0 else 0

    return {
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1_score,
        'Accuracy': accuracy,
        'Specificity': specificity,
        'False Positive Rate': false_positive_rate,
        'False Negative Rate': false_negative_rate,
        'MCC': mcc
    }

def get_mcc_per_class(file_path, file_classes_labels):

    df = pd.read_csv(file_path, sep=";")
    class_labels = np.array(df['Class Label'].tolist())
    best_match_labels = np.array(df['1st Best'].tolist())


    results_dict = {}
    # Find true and false positives, and, true and false negatives
    for i, label_i in enumerate(file_classes_labels):

        tp_idx = (class_labels == i) & (best_match_labels == i)
        fp_idx = (class_labels != i) & (best_match_labels == i)
        tn_idx = (class_labels != i) & (best_match_labels != i)
        fn_idx = (class_labels == i) & (best_match_labels != i)
        
        tp, fp = np.sum(tp_idx.astype(int)), np.sum(fp_idx.astype(int)) 
        tn, fn = np.sum(tn_idx.astype(int)), np.sum(fn_idx.astype(int))

        res_dict = calculate_metrics(tp, fp, tn, fn)
        
        # Save the MCC score
        results_dict[label_i] = res_dict['MCC']

    return results_dict

def get_performance_metrics(file_path, file_classes_labels):
    df = pd.read_csv(file_path, sep=";")
    class_labels = np.array(df['Class Label'].tolist())
    best_match_labels = np.array(df['1st Best'].tolist())

    tp, fp, tn, fn = 0, 0, 0, 0
    for i, label_i in enumerate(file_classes_labels):

        tp_idx = (class_labels == i) & (best_match_labels == i)
        fp_idx = (class_labels != i) & (best_match_labels == i)
        tn_idx = (class_labels != i) & (best_match_labels != i)
        fn_idx = (class_labels == i) & (best_match_labels != i)

        tp, fp = tp + np.sum(tp_idx.astype(int)), fp + np.sum(fp_idx.astype(int)) 
        tn, fn = tn + np.sum(tn_idx.astype(int)), fn + np.sum(fn_idx.astype(int))
    res_dict = calculate_metrics(tp, fp, tn, fn)

    print(f'Accuracy {res_dict['Accuracy']:.2f}')
    print(f'Precision {res_dict['Precision']:.2f}')
    print(f'Recall {res_dict['Recall']:.2f}')
    print(f'{res_dict['MCC']}')
    
    return res_dict

In [ ]:
# Get latex tables with MCC score per measure and class
table_entries = np.zeros((n_classes, n_measures))

for i, measure_name in enumerate(measures):
    print(f'--- Measure {measure_name}, k={downshift_threshold} ---')
    file_path = parentFolder + '/' + measure_name + '_' + common_file_name
    d1 = get_mcc_per_class(file_path, file_classes_labels)
    table_entries[:,i] = list(d1.values())

df_table = pd.DataFrame(table_entries, index=[l.replace('_', ' ') for l in file_classes_labels], columns=measures)
print(df_table.to_latex(float_format="%.4f"))

# Get latex tables with evalutaion scores per measure

eval_metrics = ['Accuracy', 'Precision', 'Recall', 'MCC']
table_entries = np.zeros((len(eval_metrics), len(measures)))

for i, measure_name in enumerate(measures):
    print(f'--- Measure {measure_name} ---')
    file_path = parentFolder + '/' + measure_name + '_' + common_file_name
    d1 = get_performance_metrics(file_path, file_classes_labels)
    table_entries[:,i] = [d1[key] for key in eval_metrics]

df_table = pd.DataFrame(table_entries, index=eval_metrics, columns=measures)
print(df_table.to_latex(float_format="%.4f"))